## Manustused

Meie eelmises näites töötlesime kõrge dimensioonilisusega sõnakoti vektoreid pikkusega `vocab_size` ning muutsime otseselt madaldimensioonilisi positsioonilisi esitusvektoreid hõredaks ühekordseks esitusviisiks. See ühekordne esitusviis ei ole mälu tõhus ning lisaks käsitletakse iga sõna sõltumatult teistest, st ühekordse kodeeringuga vektorid ei väljenda sõnade vahel ühtegi semantilist sarnasust.

Selles ühikus jätkame **News AG** andmestiku uurimist. Alustuseks laadime andmed ja võtame mõned definitsioonid eelmisest märkmikust.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## Mis on embedimine?

Mõte **embedimises** on esindada sõnu madalama dimensiooniga tihendatud vektoritega, mis mingil moel peegeldavad sõna semantilist tähendust. Hiljem arutame, kuidas ehitada tähenduslikke sõnaembedimisi, kuid praegu mõelgem embedimistele kui viisile vähendada sõnavektori dimensioone.

Niisiis, embedimise kiht võtab sisendiks sõna ja annab väljundiks etteantud `embedding_size` pikkuse vektori. Oma olemuselt on see väga sarnane `Linear` kihile, kuid ühe-kuumalt kodeeritud vektori asemel võib see võtta sisendiks sõna numbri.

Kasutades embedimise kihti võrgu esimeses kihis, saame liikuda sõnakoti mudelist üle **embedimise koti** mudelile, kus esmalt teisendame iga sõna tekstist vastavaks embedimiseks ja seejärel arvutame mõne agregatsioonifunktsiooni kõigi nende embedimiste üle, näiteks `sum`, `average` või `max`.

![Pilt, mis näitab embedimise klassifikaatorit viie järjestikuse sõna jaoks.](../../../../../translated_images/et/embedding-classifier-example.b77f021a7ee67eee.webp)

Meie klassifikaatori närvivõrk algab embedimise kihiga, seejärel agregatsioonikihiga ning selle peal lineaarse klassifikaatoriga:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### Kõrvaldamine muutuvate järjestuste pikkusega

Selle arhitektuuri tulemusena tuleb meie võrgule minibatch'id teatud viisil luua. Eelnevas üksuses, kui kasutati sõnakotti, olid kõik BoW tensorrid minibatch'is võrdse suurusega `vocab_size`, sõltumata meie tekstijärjestuse tegelikust pikkusest. Kui me liigume sõnapõhiste manuste kasutamisele, siis igas tekstinäites oleks muutuv sõnade arv ja nende näidete minibatch'idesse kombineerimisel peaksime rakendama mõnda täitmist.

Seda saab teha kasutades sama tehnikat, pakkudes andmeallikale `collate_fn` funktsiooni:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### Manuste klassifikaatori treenimine

Nüüd, kui oleme määratlenud sobiva andmete laadija, saame mudelit treenida kasutades treenimisfunktsiooni, mille me eelnevas osas määratlesime:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **Märkus**: Siin treenime ajapiirangu tõttu ainult 25 000 salvestust (vähem kui üks täielik epohh), kuid võite jätkata treenimist, kirjutada funktsiooni mitme epohhi treenimiseks ja katsetada õppemäärade parameetriga, et saavutada kõrgem täpsus. Te peaks ilmuma suutma umbes 90% täpsusele jõuda.


### EmbeddingBag kiht ja muutuva pikkusega järjendite representatsioon

Eelmises arhitektuuris pidime kõik järjendid täiendama ühesuguse pikkuseni, et need minibatch'i mahutada. See pole kõige tõhusam viis muutuva pikkusega järjendite esindamiseks – teine lähenemine oleks kasutada **offset** vektorit, mis hoiaks kõigi ühes suures vektoris salvestatud järjendite offseteid.

![Pilt, mis näitab offset-järjendi representatsiooni](../../../../../translated_images/et/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **Märkus**: Ülaloleval pildil on kujutatud tähestiku järjendit, kuid meie näites töötame sõnajärjenditega. Kuid üldine printsiip esindada järjendeid offset-vektoriga jääb samaks.

Offset-esindusega töötamiseks kasutame [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) kihti. See on sarnane `Embedding`-ga, kuid võtab sisendiks sisuvektori ja offset-vektori ning sisaldab ka keskmistamiskihte, mis võib olla `mean`, `sum` või `max`.

Siin on muudetud võrk, mis kasutab `EmbeddingBag`:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

Andmete komplekti treenimiseks peame pakkuma konverteerimisfunktsiooni, mis valmistab ette nihkevektori:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

Pane tähele, et erinevalt kõigist varasematest näidetest võtab meie võrk nüüd vastu kaks parameetrit: andmevektor ja nihkevektor, mis on erineva suurusega. Samamoodi annab ka meie andmeallikas meile 2 asemel 3 väärtust: nii tekst kui ka nihkevektorid antakse omadustena. Seetõttu peame meie treenimisfunktsiooni veidi kohandama, et sellega hakkama saada:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## Semantilised manused: Word2Vec

Meie eelmises näites õppis mudeli manustustekiht sõnu vektoripildina esindama, kuid see esindus ei omanud suurt semantilist tähendust. Oleks tore õppida selline vektoriesitus, kus sarnaste sõnade või sünonüümide vektorid oleksid mingi vektoriaalse kauguse (nt eukleidiline kaugus) poolest üksteisele lähedal.

Selle jaoks tuleb meil eelnevalt treenida oma manustust mudelit spetsiaalsel viisil suurel tekstikogumil. Üks esimesi viise semantiliste manuste treenimiseks on tuntud kui [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). See põhineb kahel põhistruktuuril sõnade jaotatud esinduse loomiseks:

 - **Jätkuv sõnapõie kotike** (Continuous bag-of-words, CBoW) — selles arhitektuuris treenime mudelit ennustama sõna ümbritseva konteksti põhjal. Antud n-gramm $(W_{-2},W_{-1},W_0,W_1,W_2)$ puhul on mudeli eesmärgiks ennustada $W_0$ sõna sõnade $(W_{-2},W_{-1},W_1,W_2)$ põhjal.
 - **Jätkuv skip-gram** on CBoW vastand. Mudel kasutab ümbritsevat konteksti sõnade akent, et ennustada praegust sõna.

CBoW on kiirem, samas skip-gram on aeglasem, kuid esindab paremini haruldasi sõnu.

![Pilt, mis näitab nii CBoW kui Skip-Gram algoritme sõnade teisendamiseks vektoriteks.](../../../../../translated_images/et/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Sõna 'neural' kõige sarnasemate sõnade katsetamiseks Google News andmestikul eelõpetatud word2vec manustega saame kasutada **gensim** raamatukogu. Allpool leiame sõnad, mis on 'neural' sõnale kõige sarnasemad.

> **Märkus:** Kui tekib sõnavektorite loomine esimest korda, võib nende allalaadimine aega võtta!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


Saame sõnast arvutada ka vektori manuseid, mida kasutada klassifitseerimismudeli treenimisel (selguse huvides näitame ainult vektori esimesed 20 komponenti):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

Semantiliste manustega seotud hea asi on see, et vektorkodeeringut saab manipuleerida, et muuta semantikat. Näiteks võime paluda leida sõna, mille vektorirepresentatsioon oleks võimalikult lähedal sõnadele *kuningas* ja *naine*, ning võimalikult kaugel sõnast *mees*:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

Nii CBoW kui Skip-Grams on „ennustavad“ embeddingud, kuna need arvestavad ainult kohalikku konteksti. Word2Vec ei kasuta globaalset konteksti.

**FastText** põhineb Word2Vecil, õppides vektorrepresentatsioone iga sõna ja selle sees leiduvate märgi n-grammide jaoks. Esinduste väärtused võetakse iga treenimissammu juures keskmistena ühte vektorisse. Kuigi see lisab eeltreenimisele palju täiendavat arvutust, võimaldab see sõna embeddingutel kodeerida ala-sõna teavet.

Teine meetod, **GloVe**, tugineb koosesinemismaatriksi ideele, kasutades närvivõrke koosesinemismaatriksi lahtidekomponeerimiseks väljendusrikkamateks ja mitte-lineaarseteks sõnavektoriteks.

Saate näitega mängida, muutes embeddingsid FastTextiks või GloVeks, kuna gensim toetab mitut erinevat sõnade embeddingu mudelit.


## Eeltreenitud manuste kasutamine PyTorchis

Saame ülaltoodud näidet muuta nii, et meie lihtkihi maatriks täidetakse eelnevalt semantiliste manustega, näiteks Word2Veciga. Peame arvestama, et eeltreenitud manustuse sõnavara ja meie teksti korpuse sõnavara ei pruugi tõenäoliselt kattuda, seega algatame puuduolevate sõnade kaalud juhuslike väärtustega:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


Nüüd treenime oma mudelit. Pane tähele, et mudeli treenimiseks kuluv aeg on oluliselt suurem kui eelnevas näites, kuna manustamiskihi suurus on suurem ja seega ka parameetrite arv palju suurem. Samuti peame selle tõttu oma mudelit võib-olla treenima rohkemate näidete peal, kui tahame vältida ületreenimist.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

Meie puhul ei näe me suurt täpsuse kasvu, mis tuleneb tõenäoliselt erinevatest sõnavaradest. 
Selle probleemi ületamiseks mitmesuguse sõnavaraga võime kasutada üht järgmistest lahendustest:
* Treenida sõna2veki mudel uuesti meie sõnavara peal
* Laadida meie andmestik koos eeltreenitud sõna2veki mudelilt pärineva sõnavaraga. Sõnavara, mida kasutada andmestiku laadimisel, saab laadimise ajal määrata.

Viimane lähenemine tundub lihtsam, eriti kuna PyTorchi `torchtext` raamistik sisaldab sisseehitatud tuge manustustele. Näiteks võime GloVe-põhise sõnavara instantsida järgmisel viisil:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Laaditud sõnavaral on järgmised põhifunktsioonid:
* `vocab.stoi` sõnastik võimaldab meil teisendada sõna selle sõnastiku indeksiks
* `vocab.itos` teeb vastupidist - teisendab numbri sõnaks
* `vocab.vectors` on embed'de vektorite massiiv, seega sõna embed'de saamiseks `s` tuleb kasutada `vocab.vectors[vocab.stoi[s]]`

Siin on näide embed'de manipuleerimisest, et demonstreerida võrrandit **lahke-mees+naine = kuninganna** (pidin koefitsienti veidi kohandama, et see töötaks):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

Selleks, et neid manuseid kasutades klassifikaatorit koolitada, peame esmalt oma andmekogu kodeerima GloVe sõnavara abil:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

Nagu eespool nägime, salvestatakse kõik vektorite embedimised `vocab.vectors` maatriksisse. See teeb nende kaalu lihtsaks laadimiseks embedimiskihi kaaludesse, kasutades lihtsat kopeerimist:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

Treenime nüüd oma mudelit ja vaatame, kas saame paremaid tulemusi:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

Üks põhjus, miks me ei näe täpsuse märkimisväärset kasvu, on see, et mõned sõnad meie andmekogumist puuduvad eelnevalt treenitud GloVe sõnavaras ning seega jäetakse need sisuliselt tähelepanuta. Selle fakti ületamiseks saame treenida oma enda embeddingud meie andmekogul.


## Kontekstuaalsed embedingud

Üks traditsiooniliste eelnevalt treenitud embeddingute, nagu Word2Vec, peamisi piiranguid on sõna tähenduse tuvastamise probleem. Kuigi eelnevalt treenitud embeddings võivad haarata mõningast sõnade tähendust kontekstis, on iga sõna võimalik tähendus kodeeritud samasse embeddingusse. See võib põhjustada probleeme allajooksu mudelites, kuna paljudel sõnadel, nagu sõnal „play“, on erinevad tähendused sõltuvalt kontekstist, milles neid kasutatakse.

Näiteks sõnal „play“ on nendes kahes erinevas lauses üsna erinev tähendus:
- Ma käisin teatris **play** vaatamas.
- John tahab oma sõpradega **play** mängida.

Ülaltoodud eelnevalt treenitud embeddingud esindavad mõlemaid tähendusi sõnast „play“ samas embeddingus. Selle piirangu ületamiseks peame looma embeddinguid, mis põhinevad **keelemudelil**, mis on treenitud suure tekstikorpuse peal ja *teab*, kuidas sõnu saab erinevates kontekstides kokku panna. Kontekstuaalsete embeddingute käsitlemine jääb selle juhendi raamidest välja, kuid me tuleme nende juurde tagasi, kui räägime keelemudelitest järgmisel alusel.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
